In [2]:
from core.dataloader import *
from core.tokenizer import *
from core.model import *
from core.model import token_to_text
from datasets import load_dataset
import torch.optim as optim

In [3]:
BATCH_SIZE = 4 # Context_len * batch_size = input per process
CONTEXT_LENGTH = 256 # How many tokens should a tokenizer see.
STRIDE = 256 # How many tokens should a dataloader process before moving to next batch.
SHUFFLE = True 
DIMENSION_OUT = 1024 # Dimension Features, Vocab Size x Dimension
TOKENIZER = Tokenizer.from_file("token_bpe.json")
VOCAB_SIZE = 8000
HEAD_NUMBER = 4

In [4]:
# dataset = load_dataset("nvidia/Nemotron-Pretraining-Dataset-sample", "Nemotron-CC-Diverse-QA", split="train")

# with open("dataset.txt", "w", encoding="utf-8") as f:
#     for row in dataset:
#         f.write(row["text"] + "\n")

In [5]:
with open("dataset.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader(
    raw_text, batch_size=BATCH_SIZE, max_length=CONTEXT_LENGTH, stride=STRIDE, shuffle=SHUFFLE, tokenizer=TOKENIZER
)

In [6]:
model = GPTModel(
    vocab_size = VOCAB_SIZE,
    context_length = CONTEXT_LENGTH,
    d_model = DIMENSION_OUT,
    n_heads = HEAD_NUMBER,
    n_layers = 8,
    )

In [7]:
start_context = "Being crazy forMe"
token = text_to_token(start_context, TOKENIZER)
print(token)
decoded = TOKENIZER.decode((token.tolist()[0]))
print(decoded)

tensor([[ 301, 5352, 4650,  357,  356,  584, 6721]])
Being crazy forMe


In [7]:
# out = generate_text(
# model=model,
# idx=encoded_tensor,
# max_new_tokens=256,
# context_size=CONTEXT_LENGTH
# )
# decoded = TOKENIZER.decode(out[0].tolist())
# print("Output:", decoded)
# print("Output length:", len(out[0]))

In [8]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 117,385,216


In [9]:
# for batch in dataloader: # Dataloader len = floor(total token / (context length * batch)). 
#     x, y = batch 
#     token_embedding = token_embeddings(x) # Given token x, convert into corresponding token's matrix
#     pos_embedding = positional_embeddings(torch.arange(CONTEXT_LENGTH)) # arange tells which tokens should be fetched for each position.
#     input_embedding = token_embedding + pos_embedding
#     out = block(input_embedding)

In [10]:
generate_and_print_sample(model=model, tokenizer=TOKENIZER, device=torch.device("cpu"), "hello") 

SyntaxError: positional argument follows keyword argument (57759298.py, line 1)

In [ ]:
generate_and_print_sample(model, TOKENIZER, torch.device("cpu"), "who are you")

In [ ]:
token_to_text(generate_text(model, text_to_token("hello", TOKENIZER), 124, 124), TOKENIZER)

In [ ]:
print(model.positional_embeddings.weight.shape[0])

In [ ]:
model_training(model=model, train_loader=dataloader, eval_loader=dataloader, optimizer=optim.SGD(model.parameters(), lr=0.01, momentum=0.9), device=torch.device("cpu"), eval_freq= 5, eval_iter=5, start_context="hello", num_epochs=1,tokenizer=TOKENIZER)

Ep 1 (Step 000000): Train loss 9.143, Val loss 9.138
Ep 1 (Step 000005): Train loss 8.903, Val loss 8.851
Ep 1 (Step 000010): Train loss 8.431, Val loss 8.511
